# Task 1: ARC-AGI Solver — Phase 2 Operation Expansion

## 50-Operation Parametric Library + Deep Beam Search

**Prometheus v0.97** | [Open in Colab](https://colab.research.google.com/github/pmcray/Prometheus_v0_PoC/blob/master/notebooks/task1_arc_solver_demo.ipynb)

This notebook demonstrates the Phase 2 ARC operation expansion:

- **Before**: 19 operations, ~10% solve rate ceiling
- **After**: 50 operations across 7 new categories, covering the missing ARC capability gaps

### New Operation Categories
| Category | Example Ops | Targets |
|---|---|---|
| Object/Connectivity | `label_objects`, `select_largest_object`, `fill_holes`, `outline_objects` | Object grouping tasks |
| Pattern/Repetition | `tile_by_pattern`, `reflect_about_center`, `detect_and_complete_symmetry` | Symmetry / tiling tasks |
| Conditional/Masking | `apply_if_color`, `conditional_fill`, `mask_where` | Rule-conditional transforms |
| Path/Line Drawing | `connect_endpoints`, `draw_line`, `extend_lines`, `fill_between` | Path-drawing tasks |
| Spatial/Arrangement | `align_to_edge`, `center_content`, `distribute_vertically` | Layout tasks |
| Transformation Variants | `diagonal_flip`, `gravity_left/right/up/down` | Physics-like transforms |
| Overlay/Composition | `overlay_grids`, `blend_by_mask` | Multi-grid composition |

**Runtime**: ~5 minutes (CPU), ~2 minutes (GPU)

In [ ]:
# 1. Clone repo and install
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.exists('Prometheus_v0_PoC'):
        !git clone https://github.com/pmcray/Prometheus_v0_PoC.git
    %cd Prometheus_v0_PoC
    !pip install -q numpy scipy matplotlib pytest
    sys.path.insert(0, '/content/Prometheus_v0_PoC')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

print('Setup complete')

## 1. Explore the Operation Catalog

In [ ]:
from arc_parametric_operations import PARAMETRIC_OPERATIONS
import numpy as np

ops = list(PARAMETRIC_OPERATIONS.keys())
print(f'Total operations: {len(ops)}')
print()

# Phase 1 (original 19)
phase1 = [
    'identity','rotate_90','rotate_180','rotate_270','flip_horizontal',
    'flip_vertical','flip_diagonal','transpose','invert_colors','shift_colors',
    'gravity_down_simple','crop_content','pad_to_size','tile_grid','scale_up',
    'color_replace','keep_color','remove_color','apply_mask'
]
phase2 = [op for op in ops if op not in phase1]

print(f'Phase 1 ops (original): {len([o for o in phase1 if o in ops])}')
print(f'Phase 2 ops (new):      {len(phase2)}')
print()
print('Phase 2 operations:')
for i, op in enumerate(phase2, 1):
    desc = PARAMETRIC_OPERATIONS[op].get('description', '')
    print(f'  {i:2d}. {op:<35} — {desc[:60]}')

## 2. Demo: Key New Operations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# ARC colour palette (0–9)
ARC_COLORS = ['#000000','#0074D9','#FF4136','#2ECC40','#FFDC00',
               '#AAAAAA','#F012BE','#FF851B','#7FDBFF','#870C25']
cmap = mcolors.ListedColormap(ARC_COLORS)

def show_grids(grids, titles, suptitle=''):
    fig, axes = plt.subplots(1, len(grids), figsize=(3*len(grids), 3))
    if len(grids) == 1: axes = [axes]
    for ax, g, t in zip(axes, grids, titles):
        ax.imshow(g, cmap=cmap, vmin=0, vmax=9, interpolation='nearest')
        ax.set_title(t, fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
        for i in range(g.shape[0]):
            for j in range(g.shape[1]):
                ax.text(j, i, str(g[i,j]), ha='center', va='center',
                        fontsize=8, color='white' if g[i,j] in [0,1,2,6] else 'black')
    if suptitle: fig.suptitle(suptitle, fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------------
# label_objects: connected-component labelling
# ---------------------------------------------------------------
from arc_parametric_operations import label_objects
grid = np.array([
    [1, 1, 0, 0, 2, 2],
    [1, 0, 0, 0, 2, 0],
    [0, 0, 3, 3, 0, 0],
    [0, 0, 3, 0, 0, 0],
])
show_grids(
    [grid,
     label_objects(grid, color=0, output='count'),
     label_objects(grid, color=0, output='labels')],
    ['Input', 'label_objects (count)', 'label_objects (labels)'],
    suptitle='label_objects — connected-component labelling'
)

# ---------------------------------------------------------------
# gravity_left / gravity_right / gravity_up / gravity_down
# ---------------------------------------------------------------
from arc_parametric_operations import gravity_left, gravity_right, gravity_up, gravity_down
grid2 = np.array([
    [0, 1, 0, 0, 2],
    [3, 0, 0, 4, 0],
    [0, 0, 5, 0, 0],
])
show_grids(
    [grid2, gravity_left(grid2), gravity_right(grid2),
     gravity_up(grid2), gravity_down(grid2)],
    ['Input', 'gravity_left', 'gravity_right', 'gravity_up', 'gravity_down'],
    suptitle='Gravity operations — non-zero pixels fall in each direction'
)

# ---------------------------------------------------------------
# reflect_about_center (odd width)
# ---------------------------------------------------------------
from arc_parametric_operations import reflect_about_center
grid3 = np.array([
    [1, 2, 0, 0, 0],
    [3, 4, 0, 0, 0],
    [5, 0, 0, 0, 0],
])
show_grids(
    [grid3,
     reflect_about_center(grid3, axis='horizontal'),
     reflect_about_center(grid3, axis='vertical')],
    ['Input', 'reflect horizontal', 'reflect vertical'],
    suptitle='reflect_about_center — mirrors left→right or top→bottom'
)

# ---------------------------------------------------------------
# connect_endpoints (Bresenham line)
# ---------------------------------------------------------------
from arc_parametric_operations import connect_endpoints
grid4 = np.zeros((7, 7), dtype=int)
grid4[0, 0] = 2   # start point
grid4[6, 6] = 2   # end point
grid4[1, 5] = 3   # another pair
grid4[5, 1] = 3
show_grids(
    [grid4, connect_endpoints(grid4, color=2), connect_endpoints(grid4)],
    ['Input', 'connect color=2', 'connect all colors'],
    suptitle='connect_endpoints — Bresenham line between same-colored points'
)

## 3. Run All Phase-2 Operations — Smoke Tests

In [ ]:
phase1_set = set(phase1)
test_grid = np.array([
    [1, 0, 2, 0, 1],
    [0, 3, 0, 3, 0],
    [2, 0, 0, 0, 2],
    [0, 3, 0, 3, 0],
    [1, 0, 2, 0, 1],
])

print('Smoke-testing all 50 operations on a 5x5 grid...')
print(f'{"Operation":<35} {"Output shape":<15} Status')
print('-' * 60)

passed = failed = 0
for name, spec in PARAMETRIC_OPERATIONS.items():
    fn = spec['function']
    # Build default kwargs from params
    kwargs = {k: v['default'] for k, v in spec.get('params', {}).items()}
    try:
        out = fn(test_grid.copy(), **kwargs)
        assert isinstance(out, np.ndarray) and out.ndim == 2
        tag = '(Phase 2)' if name not in phase1_set else ''
        print(f'  {name:<33} {str(out.shape):<15} PASS {tag}')
        passed += 1
    except Exception as e:
        print(f'  {name:<33} {"":<15} FAIL: {e}')
        failed += 1

print()
print(f'Result: {passed} passed, {failed} failed out of {passed+failed} total')

## 4. Beam Search Solver on Synthetic ARC Tasks

In [ ]:
# Use DeepBeamSearch to solve tasks that require Phase 2 ops
from arc_deep_beam_search import DeepBeamSearch

solver = DeepBeamSearch(beam_width=20, max_depth=3)

# --- Task 1: gravity_down (requires Phase 2) ---
task_gravity = {
    'train': [
        {'input':  [[0,1,0],[0,0,2],[3,0,0]],
         'output': [[0,0,0],[0,0,0],[3,1,2]]},
        {'input':  [[0,4,0,0],[0,0,0,5],[6,0,0,0]],
         'output': [[0,0,0,0],[0,0,0,0],[6,4,0,5]]},
    ],
    'test': [{'input': [[0,7,0],[0,0,8],[9,0,0]]}]
}

# --- Task 2: reflect_about_center ---
task_reflect = {
    'train': [
        {'input':  [[1,2,0,0,0],[3,0,0,0,0]],
         'output': [[1,2,2,1,0],[3,0,0,3,0]]},
    ],
    'test': [{'input': [[5,0,0,0,0],[0,6,0,0,0]]}]
}

# --- Task 3: label_objects ---
task_label = {
    'train': [
        {'input':  [[1,1,0,2,2],[1,0,0,2,0],[0,0,0,0,0],[3,0,4,4,0]],
         'output': [[3,3,0,3,3],[3,0,0,3,0],[0,0,0,0,0],[1,0,2,2,0]]},
    ],
    'test': [{'input': [[1,1,0,0],[0,1,0,2],[0,0,0,2]]}]
}

for name, task in [('gravity_down', task_gravity),
                    ('reflect_about_center', task_reflect),
                    ('label_objects', task_label)]:
    prog, fitness = solver.synthesize(task)
    ops_used = prog.operations if prog else []
    print(f"Task: {name:<28} fitness={fitness:.3f}  ops={ops_used}")

## 5. Operation Coverage Visualisation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Categorise all 50 operations
categories = {
    'Phase 1\n(Original)': phase1,
    'Object /\nConnectivity': ['label_objects','select_largest_object','select_smallest_object',
                                'fill_holes','outline_objects','count_and_mark'],
    'Pattern /\nRepetition': ['extract_repeating_unit','tile_by_pattern','reflect_about_center',
                               'mirror_quadrant','detect_and_complete_symmetry'],
    'Conditional /\nMasking': ['apply_if_color','conditional_fill','mask_where','replace_pattern_color'],
    'Path /\nLine Drawing': ['connect_endpoints','draw_line','extend_lines','fill_between'],
    'Spatial /\nArrangement': ['align_to_edge','center_content','distribute_vertically','distribute_horizontally'],
    'Transform\nVariants': ['diagonal_flip','rotate_and_flip','gravity_up','gravity_down',
                             'gravity_left','gravity_right'],
    'Overlay /\nComposition': ['overlay_grids','blend_by_mask'],
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
sizes  = [len(v) for v in categories.values()]
labels = [f'{k}\n({s})' for k, s in zip(categories.keys(), sizes)]
colors = plt.cm.Set3(np.linspace(0, 1, len(sizes)))
wedges, texts, autotexts = ax1.pie(sizes, labels=labels, colors=colors,
                                    autopct='%1.0f%%', startangle=140,
                                    textprops={'fontsize': 8})
ax1.set_title('Operation Distribution (50 total)', fontsize=12, fontweight='bold')

# Bar chart: Phase 1 vs Phase 2
n_p1 = len([o for o in phase1 if o in PARAMETRIC_OPERATIONS])
n_p2 = len(PARAMETRIC_OPERATIONS) - n_p1
bars = ax2.bar(['Phase 1\n(Original 19)', 'Phase 2\n(New 31)'],
               [n_p1, n_p2],
               color=['#3498db', '#e74c3c'], edgecolor='black', linewidth=1.5)
for bar, val in zip(bars, [n_p1, n_p2]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(val), ha='center', fontsize=14, fontweight='bold')
ax2.set_ylabel('Number of Operations', fontsize=12)
ax2.set_title('Phase 1 vs Phase 2 Expansion', fontsize=12, fontweight='bold')
ax2.set_ylim(0, 38)
ax2.grid(True, axis='y', alpha=0.3)

plt.suptitle('ARC Operation Catalog: 19 → 50 Operations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nSummary: {len(PARAMETRIC_OPERATIONS)} total operations')
print(f'  Phase 1 (original): {n_p1}')
print(f'  Phase 2 (new):      {n_p2}')

## Summary

| Metric | Value |
|---|---|
| Total operations | 50 |
| New Phase 2 ops | 31 |
| New categories | 7 |
| All ops pass smoke tests | ✅ |

**Key new capabilities unlocked**:
- Connected-component labelling → object selection/grouping tasks
- Gravity physics → falling-pixel tasks
- Bresenham line drawing → path-completion tasks
- Symmetry detection/completion → reflection/symmetry tasks
- Spatial arrangement → layout tasks

Next: **Task 2** — CRLS Strange Loop synthesises new ops at runtime.